# SwiftSQL V2 (opt2): 批量并行评测 + RPO 稳态多轮 DPO + API 自修复闭环评测

> **针对上一轮成绩表的精准升级**：
> 1. **彻底治好 DPO 塌缩**：采用 **`rpo_alpha=0.5`（DPO + NLL 锚定）+ 温和学习率（`1e-5 → 5e-6`）+ 强 KL 约束（`beta=0.20 → 0.25`）+ 每轮 1 epoch**，稳守 `syntax ≥ 99%` 与 `refusal ≥ 94%` 同时把 `EX` 往上冲；
> 2. **极速输出**：配合 `opt2` 极简 Auto-Think，平均输出字数与延迟目标追平并反超 Base；
> 3. **新增 `Best+Repair` 实战列**：在验证集上模拟 API 网关对语法/执行报错自动触发 1 次模型自修复，量化端到端真实可用率。


In [ ]:
# 1. 挂载 Google Drive、拉取最新代码并安装依赖
import inspect, os, sys, time, torch, yaml
from pathlib import Path
from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive", force_remount=True)
else:
    print("Google Drive already mounted at /content/drive")

DRIVE_DIR = "/content/drive/MyDrive/swift-sql"
REPO_DIR = "/content/swift-sql"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/2422725977/swift-sql.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

!pip install -q --upgrade unsloth trl datasets sqlglot pyyaml
!pip install -q --no-deps -e {REPO_DIR}
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

PROFILE = "3b"  # 可选 "3b" (opt2, bf16高速) 或 "7b" (opt2_7b, 高精度双开赛马)
CFG_FILE = "train_config_7b.yaml" if PROFILE == "7b" else "train_config.yaml"
with open(f"{REPO_DIR}/configs/{CFG_FILE}", "r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)
LOAD_IN_4BIT = CFG["model"].get("load_in_4bit", True)
print(f"Active Profile={PROFILE} | Model={CFG['model']['base_model']} | 4bit={LOAD_IN_4BIT}")

RUN_TAG = CFG["project"].get("run_tag", "opt2")
RUN_DIR = Path(DRIVE_DIR) / RUN_TAG
print(f"Active run directory: {RUN_DIR}")

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Ready: {name} ({vram:.1f} GB VRAM) | bf16={torch.cuda.is_bf16_supported()}")
else:
    raise RuntimeError("未检测到 GPU！请切换到 L4 或 A100 运行时。")


In [ ]:
# 2. 长度分桶批量推理引擎 + 基准评测（Base vs opt2 V1_SFT）
from tqdm.auto import tqdm
from unsloth import FastLanguageModel
from swiftsql.data_pipeline import load_jsonl, save_jsonl
from swiftsql.dialect import transpile_sql, validate_sql
from swiftsql.evaluation import evaluate_predictions, execute_sqlite_sandbox
from swiftsql.prompts import build_repair_messages, parse_model_response

EVAL_PATH = RUN_DIR / "data" / "eval.jsonl"
TRAIN_PATH = RUN_DIR / "data" / "train_sft.jsonl"
eval_records = load_jsonl(EVAL_PATH)
train_records = load_jsonl(TRAIN_PATH)
dpo_cfg = CFG["dpo"]
print(f"Loaded [{RUN_TAG}] eval={len(eval_records)}, train={len(train_records)}")

def _prepare_tokenizer_for_batch(tok):
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok

def batch_generate(
    model,
    tok,
    prompts,
    *,
    batch_size=16,
    max_new_tokens=256,
    do_sample=False,
    temperature=0.0,
    top_p=0.9,
    num_return_sequences=1,
    desc="Batched inference",
):
    _prepare_tokenizer_for_batch(tok)
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    results = [None] * len(prompts)

    def _run_chunk(sub_idxs):
        batch_prompts = [prompts[i] for i in sub_idxs]
        inputs = tok(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1536,
        ).to("cuda")
        in_len = inputs["input_ids"].shape[1]
        gen_kwargs = {
            "max_new_tokens": max_new_tokens,
            "do_sample": do_sample,
            "num_return_sequences": num_return_sequences,
            "pad_token_id": tok.pad_token_id,
            "eos_token_id": tok.eos_token_id,
        }
        if do_sample:
            gen_kwargs["temperature"] = temperature
            gen_kwargs["top_p"] = top_p
        outs = model.generate(**inputs, **gen_kwargs)
        decoded = tok.batch_decode(outs[:, in_len:], skip_special_tokens=True)
        if num_return_sequences == 1:
            for idx, txt in zip(sub_idxs, decoded):
                results[idx] = txt.strip()
        else:
            for pos, idx in enumerate(sub_idxs):
                chunk = decoded[pos * num_return_sequences : (pos + 1) * num_return_sequences]
                results[idx] = [t.strip() for t in chunk]

    pbar = tqdm(range(0, len(order), batch_size), desc=desc)
    with torch.inference_mode():
        for start in pbar:
            idxs = order[start : start + batch_size]
            queue = [idxs]
            while queue:
                sub = queue.pop(0)
                try:
                    _run_chunk(sub)
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache()
                    if len(sub) == 1:
                        raise
                    mid = len(sub) // 2
                    queue.insert(0, sub[mid:])
                    queue.insert(0, sub[:mid])
    return results

def run_benchmark(model_name_or_path, records, max_samples=None, batch_size=None, label=None, return_preds=False):
    max_samples = max_samples or dpo_cfg.get("eval_samples", 300)
    batch_size = batch_size or dpo_cfg.get("eval_batch_size", 16)
    subset = records[:max_samples]
    tag = label or Path(model_name_or_path).name

    m, tok = FastLanguageModel.from_pretrained(
        model_name=model_name_or_path,
        max_seq_length=CFG["model"]["max_seq_length"],
        dtype=None,
        load_in_4bit=LOAD_IN_4BIT,
    )
    FastLanguageModel.for_inference(m)
    prompts = [
        tok.apply_chat_template([x for x in r["messages"] if x["role"] != "assistant"], tokenize=False, add_generation_prompt=True)
        for r in subset
    ]

    t_all = time.perf_counter()
    preds = batch_generate(m, tok, prompts, batch_size=batch_size, do_sample=False, desc=f"Eval {tag}")
    total_s = time.perf_counter() - t_all

    # True single-request latency probe on 20 samples
    single_lats = []
    with torch.inference_mode():
        for p in prompts[:20]:
            inp = tok([p], return_tensors="pt", truncation=True, max_length=1536).to("cuda")
            t0 = time.perf_counter()
            m.generate(**inp, max_new_tokens=256, do_sample=False, pad_token_id=tok.pad_token_id)
            single_lats.append((time.perf_counter() - t0) * 1000.0)

    report = evaluate_predictions(subset, preds, latencies_ms=single_lats)
    report.pop("details", None)
    report["throughput_qps"] = round(len(subset) / max(total_s, 1e-6), 2)
    del m
    torch.cuda.empty_cache()
    if return_preds:
        return report, preds
    return report

REPORTS = {}
PREDS_CACHE = {}
REPORTS["Base"] = run_benchmark(CFG["model"]["base_model"], eval_records, label="Base-3B")
REPORTS["V1_SFT"], PREDS_CACHE["V1_SFT"] = run_benchmark(
    str(RUN_DIR / "models" / "v1_sft_lora"), eval_records, label="V1-SFT", return_preds=True
)

def print_comparison(reports_dict):
    names = list(reports_dict.keys())
    header = f"{'Metric':22s} | " + " | ".join(f"{n:>11s}" for n in names)
    print("\n" + "=" * len(header))
    print(header)
    print("-" * len(header))
    keys = [
        "json_valid_rate",
        "syntax_valid_rate",
        "execution_accuracy",
        "refusal_f1",
        "avg_output_chars",
        "avg_latency_ms",
        "throughput_qps",
    ]
    for k in keys:
        row = f"{k:22s} | " + " | ".join(f"{str(reports_dict[n].get(k)):>11s}" for n in names)
        print(row)
    print("=" * len(header))

print_comparison(REPORTS)


In [ ]:
# 3. RPO 稳态多轮迭代执行引导 DPO（lr=1e-5 -> 5e-6, beta=0.20 -> 0.25, rpo_alpha=0.5）
import random
from datasets import Dataset
from unsloth import PatchDPOTrainer
from trl import DPOConfig, DPOTrainer
from swiftsql.dpo_builder import build_dpo_pair

PatchDPOTrainer()

DPO_ROUNDS = dpo_cfg.get("rounds", 2)
NUM_PROMPTS_PER_ROUND = dpo_cfg.get("num_prompts_per_round", 1200)
CANDS_PER_ROUND = dpo_cfg.get("candidates_per_round", [4, 6])
LRS_PER_ROUND = dpo_cfg.get("round_learning_rates", [1e-5, 5e-6])
BETAS_PER_ROUND = dpo_cfg.get("round_betas", [0.20, 0.25])
RPO_ALPHA = dpo_cfg.get("rpo_alpha", 0.5)
MAX_GRAD_NORM = dpo_cfg.get("max_grad_norm", 0.5)
SAMPLE_BATCH_SIZE = dpo_cfg.get("sample_batch_size", 8)
FORCE_RESAMPLE = False

rng = random.Random(CFG["project"]["seed"])
exec_hard = [r for r in train_records if r["meta"].get("executable_context") and r["complexity"] in ("medium", "complex")]
exec_easy = [r for r in train_records if r["meta"].get("executable_context") and r["complexity"] == "simple"]
refusal_all = [r for r in train_records if r["task_type"] == "refusal"]
rng.shuffle(exec_hard)
rng.shuffle(exec_easy)
rng.shuffle(refusal_all)
print(f"Pool ready: hard_exec={len(exec_hard)}, easy_exec={len(exec_easy)}, refusal={len(refusal_all)}")

current_model_path = str(RUN_DIR / "models" / "v1_sft_lora")
MODEL_PATHS = {"V1_SFT": current_model_path}

dpo_sig_params = inspect.signature(DPOConfig.__init__).parameters
supports_rpo = "rpo_alpha" in dpo_sig_params
print(f"DPOConfig supports rpo_alpha={supports_rpo} (using rpo_alpha={RPO_ALPHA})")

for r_idx in range(1, DPO_ROUNDS + 1):
    round_tag = f"V2_R{r_idx}"
    n_cand = CANDS_PER_ROUND[min(r_idx - 1, len(CANDS_PER_ROUND) - 1)]
    lr = LRS_PER_ROUND[min(r_idx - 1, len(LRS_PER_ROUND) - 1)]
    beta = BETAS_PER_ROUND[min(r_idx - 1, len(BETAS_PER_ROUND) - 1)]
    pair_path = RUN_DIR / "data" / f"dpo_pairs_round{r_idx}.jsonl"
    out_lora_dir = str(RUN_DIR / "models" / f"v2_dpo_r{r_idx}_lora")

    print(f"\n================ Starting {round_tag} (cands={n_cand}, lr={lr}, beta={beta}, rpo_alpha={RPO_ALPHA}) ================")

    hard_ratio = 0.60 if r_idx == 1 else 0.75
    n_hard = int(NUM_PROMPTS_PER_ROUND * hard_ratio)
    n_easy = NUM_PROMPTS_PER_ROUND - n_hard
    h_start = ((r_idx - 1) * n_hard) % max(len(exec_hard), 1)
    e_start = ((r_idx - 1) * n_easy) % max(len(exec_easy), 1)
    ref_start = ((r_idx - 1) * 180) % max(len(refusal_all), 1)

    pool = (
        (exec_hard[h_start:] + exec_hard[:h_start])[:n_hard]
        + (exec_easy[e_start:] + exec_easy[:e_start])[:n_easy]
        + (refusal_all[ref_start:] + refusal_all[:ref_start])[:180]
    )

    if pair_path.exists() and not FORCE_RESAMPLE:
        dpo_pairs = load_jsonl(pair_path)
        print(f"[{round_tag}] Loaded cached DPO pairs: {len(dpo_pairs)}")
    else:
        m, tok = FastLanguageModel.from_pretrained(
            model_name=current_model_path,
            max_seq_length=CFG["model"]["max_seq_length"],
            dtype=None,
            load_in_4bit=LOAD_IN_4BIT,
        )
        FastLanguageModel.for_inference(m)
        prompts = [
            tok.apply_chat_template([x for x in rec["messages"] if x["role"] != "assistant"], tokenize=False, add_generation_prompt=True)
            for rec in pool
        ]
        eff_bs = max(2, SAMPLE_BATCH_SIZE * 4 // n_cand)
        all_cands = batch_generate(
            m,
            tok,
            prompts,
            batch_size=eff_bs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.70,
            top_p=0.92,
            num_return_sequences=n_cand,
            desc=f"{round_tag} Sampling ({len(pool)}x{n_cand}, bs={eff_bs})",
        )
        dpo_pairs = []
        for rec, cands in zip(pool, all_cands):
            pair = build_dpo_pair(rec, cands)
            if pair is not None:
                dpo_pairs.append(pair)

        save_jsonl(dpo_pairs, pair_path)
        del m
        torch.cuda.empty_cache()
        print(f"[{round_tag}] Built {len(dpo_pairs)} verified DPO pairs -> {pair_path}")

    if len(dpo_pairs) < 40:
        print(f"[{round_tag}] Too few preference pairs ({len(dpo_pairs)}), stopping early.")
        break

    rng.shuffle(dpo_pairs)
    val_size = max(20, min(120, int(len(dpo_pairs) * 0.10)))
    val_pairs = dpo_pairs[:val_size]
    trn_pairs = dpo_pairs[val_size:]

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=current_model_path,
        max_seq_length=CFG["model"]["max_seq_length"],
        dtype=None,
        load_in_4bit=LOAD_IN_4BIT,
    )
    for _n, _p in model.named_parameters():
        if "lora_" in _n:
            _p.requires_grad = True

    def _fmt(pairs):
        return Dataset.from_list([
            {
                "prompt": tokenizer.apply_chat_template(p["prompt"], tokenize=False, add_generation_prompt=True),
                "chosen": p["chosen"][0]["content"] + tokenizer.eos_token,
                "rejected": p["rejected"][0]["content"] + tokenizer.eos_token,
            }
            for p in pairs
        ])

    train_dpo_ds = _fmt(trn_pairs)
    eval_dpo_ds = _fmt(val_pairs)
    use_bf16 = torch.cuda.is_bf16_supported()

    dpo_args_dict = dict(
        output_dir=str(RUN_DIR / "checkpoints" / f"v2_dpo_r{r_idx}"),
        beta=beta,
        learning_rate=lr,
        num_train_epochs=dpo_cfg.get("num_train_epochs", 1),
        per_device_train_batch_size=dpo_cfg["per_device_train_batch_size"],
        gradient_accumulation_steps=dpo_cfg["gradient_accumulation_steps"],
        max_grad_norm=MAX_GRAD_NORM,
        warmup_ratio=0.10,
        lr_scheduler_type="cosine",
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=30,
        save_steps=30,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        save_total_limit=2,
        fp16=not use_bf16,
        bf16=use_bf16,
        optim="adamw_8bit",
        max_length=CFG["model"]["max_seq_length"],
        max_prompt_length=1536,
        report_to="none",
    )
    if supports_rpo:
        dpo_args_dict["rpo_alpha"] = RPO_ALPHA

    trainer = DPOTrainer(
        model=model,
        ref_model=None,
        processing_class=tokenizer,
        train_dataset=train_dpo_ds,
        eval_dataset=eval_dpo_ds,
        args=DPOConfig(**dpo_args_dict),
    )
    trainer.train()

    model.save_pretrained(out_lora_dir)
    tokenizer.save_pretrained(out_lora_dir)
    del model, trainer
    torch.cuda.empty_cache()

    REPORTS[round_tag], PREDS_CACHE[round_tag] = run_benchmark(
        out_lora_dir, eval_records, label=round_tag, return_preds=True
    )
    MODEL_PATHS[round_tag] = out_lora_dir
    # 只有本轮没有发生退化时才接力作为下一轮起点，否则从当前最优重新出发
    def _quick_score(rep):
        return 0.55 * (rep.get("execution_accuracy") or 0) + 0.25 * (rep.get("syntax_valid_rate") or 0) + 0.20 * (rep.get("refusal_f1") or 0)
    best_so_far = max(MODEL_PATHS.keys(), key=lambda k: _quick_score(REPORTS[k]))
    current_model_path = MODEL_PATHS[best_so_far]
    print(f"[{round_tag}] Best checkpoint so far: {best_so_far} -> next round starts from {current_model_path}")
    print_comparison(REPORTS)


In [ ]:
# 4. 自动择优 + 实战闭环评测（Best + 1次API报错自修复）+ 导出最终 GGUF
def composite_score(rep):
    ex = rep.get("execution_accuracy") or 0.0
    syn = rep.get("syntax_valid_rate") or 0.0
    ref = rep.get("refusal_f1") or 0.0
    js = rep.get("json_valid_rate") or 0.0
    return 0.55 * ex + 0.20 * syn + 0.15 * ref + 0.10 * js

candidates = {k: v for k, v in REPORTS.items() if k in MODEL_PATHS}
best_tag = max(candidates.keys(), key=lambda k: composite_score(candidates[k]))
best_path = MODEL_PATHS[best_tag]
print(f">>> Best single-pass checkpoint: {best_tag} ({best_path}) | composite={composite_score(REPORTS[best_tag]):.4f}")

# 模拟 API 网关的 1-shot Self-Repair 护栏（对语法错或执行报错样本自动修复 1 次）
best_model, best_tok = FastLanguageModel.from_pretrained(
    model_name=best_path,
    max_seq_length=CFG["model"]["max_seq_length"],
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)
FastLanguageModel.for_inference(best_model)

eval_subset = eval_records[: dpo_cfg.get("eval_samples", 300)]
repaired_preds = list(PREDS_CACHE[best_tag])
repair_indices, repair_prompts = [], []

for idx, (rec, raw_pred) in enumerate(zip(eval_subset, repaired_preds)):
    meta = rec.get("meta", {})
    if not meta.get("answerable", True):
        continue
    d = meta.get("dialect", "postgresql")
    parsed = parse_model_response(raw_pred, default_dialect=d)
    if not parsed["answerable"] or not parsed["sql"]:
        continue
    val = validate_sql(parsed["sql"], dialect=d, require_read_only=True)
    err_msg = None
    if not val.valid:
        err_msg = val.error
    elif meta.get("executable_context"):
        pred_sqlite = transpile_sql(parsed["sql"], source_dialect=d, target_dialect="sqlite")
        if pred_sqlite:
            out = execute_sqlite_sandbox(meta["executable_context"], pred_sqlite)
            if not out.success:
                err_msg = out.error
    if err_msg:
        msgs = build_repair_messages(
            schema=meta["schema"],
            question=meta["question"],
            broken_sql=parsed["sql"],
            error_message=err_msg,
            dialect=d,
        )
        repair_indices.append(idx)
        repair_prompts.append(best_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))

print(f"Triggered 1-shot self-repair on {len(repair_indices)} / {len(eval_subset)} samples.")
if repair_prompts:
    fixes = batch_generate(
        best_model, best_tok, repair_prompts, batch_size=16, do_sample=False, desc="1-shot Self-Repair"
    )
    for idx, fix in zip(repair_indices, fixes):
        repaired_preds[idx] = fix

repair_report = evaluate_predictions(eval_subset, repaired_preds)
repair_report.pop("details", None)
repair_report["avg_latency_ms"] = REPORTS[best_tag].get("avg_latency_ms")
repair_report["throughput_qps"] = REPORTS[best_tag].get("throughput_qps")
REPORTS["Best+Repair"] = repair_report
print_comparison(REPORTS)

GGUF_V2_DIR = str(RUN_DIR / "exports" / "v2_best_gguf")
os.makedirs(GGUF_V2_DIR, exist_ok=True)
for q in CFG["export"]["quantizations"]:
    print(f"Exporting {best_tag} GGUF ({q}) -> {GGUF_V2_DIR} ...")
    best_model.save_pretrained_gguf(GGUF_V2_DIR, best_tok, quantization_method=q)

!ls -lh {GGUF_V2_DIR}


In [ ]:
# 释放 Colab Pro GPU 配额（先刷新 Google Drive 缓存，防止权重/GGUF 丢数据）
import os, time
from google.colab import drive, runtime

AUTO_UNASSIGN = True  # 如需保留当前会话继续调试，可改为 False

if AUTO_UNASSIGN:
    print("Syncing and flushing Google Drive before releasing GPU runtime...")
    os.sync()
    time.sleep(5)
    drive.flush_and_unmount()
    print("Google Drive flushed. Releasing Colab GPU runtime now to save compute units!")
    runtime.unassign()
else:
    print("AUTO_UNASSIGN=False: 保持当前会话。用完后请手动点击右上角「断开连接并删除运行时」以停止扣费。")
